In [1]:
import pandas as pd
import numpy as np

from sdv.metadata import SingleTableMetadata
from sdv.single_table import TVAESynthesizer

print("Libraries loaded successfully.")

Libraries loaded successfully.


1. Load the preprocessed Cert dataset

In [2]:
data = pd.read_csv(
    r"C:\Users\Admin\OneDrive - Sol Plaatje University\Documents\cert_preprocessed.csv"
)

print("Dataset Shape:", data.shape)

data.head()

Dataset Shape: (248301, 17)


,timestamp,user_id,location,action,hour,day,month,weekday,after_hours,employee_name,email,role,business_unit,functional_unit,department,team,supervisor
0,2010-01-02 06:49:00,NGF0157,PC-6056,logon,6,2,1,Saturday,0,Nissim Gil French,Nissim.Gil.French@dtaa.com,FinancialAnalyst,1,4 - Finance,3 - FinancialPlanning,Unknown,Libby Rosalyn Richard
1,2010-01-02 06:50:00,LRR0148,PC-4275,logon,6,2,1,Saturday,0,Libby Rosalyn Richard,Libby.Rosalyn.Richard@dtaa.com,Manager,1,4 - Finance,3 - FinancialPlanning,Unknown,Thomas Vladimir Stokes
2,2010-01-02 06:53:04,LRR0148,PC-4124,logon,6,2,1,Saturday,0,Libby Rosalyn Richard,Libby.Rosalyn.Richard@dtaa.com,Manager,1,4 - Finance,3 - FinancialPlanning,Unknown,Thomas Vladimir Stokes
3,2010-01-02 07:00:00,IRM0931,PC-7188,logon,7,2,1,Saturday,0,Ignatius Reese Morton,Ignatius.Reese.Morton@dtaa.com,ComputerProgrammer,1,2 - ResearchAndEngineering,2 - SoftwareManagement,6 - Software,Caleb Mason Ball
4,2010-01-02 07:00:00,MOH0273,PC-6699,logon,7,2,1,Saturday,0,Macaulay Otto Hopkins,Macaulay.Otto.Hopkins@dtaa.com,HardwareEngineer,1,2 - ResearchAndEngineering,3 - Engineering,4 - SystemsEngineering,Gage Kaden Odonnell


2. Filling in the missing values

In [3]:
categorical_cols = data.select_dtypes(include="object").columns

for col in categorical_cols:
    data[col] = data[col].fillna("Unknown")

numeric_cols = data.select_dtypes(include=np.number).columns

for col in numeric_cols:
    data[col] = data[col].fillna(data[col].median())

print(data.isnull().sum())

timestamp          0
user_id            0
location           0
action             0
hour               0
day                0
month              0
weekday            0
after_hours        0
employee_name      0
email              0
role               0
business_unit      0
functional_unit    0
department         0
team               0
supervisor         0
dtype: int64


3. Convert timestamp to datetime

In [4]:
data["timestamp"] = pd.to_datetime(data["timestamp"])

print(data.dtypes)

timestamp          datetime64[ns]
user_id                    object
location                   object
action                     object
hour                        int64
day                         int64
month                       int64
weekday                    object
after_hours                 int64
employee_name              object
email                      object
role                       object
business_unit               int64
functional_unit            object
department                 object
team                       object
supervisor                 object
dtype: object


4. creation of metadata

In [5]:
metadata = SingleTableMetadata()

metadata.detect_from_dataframe(data)

print("Metadata detected successfully.")

Metadata detected successfully.


5. reducing the training data

In [6]:
training_data = data.sample(
    n=50000,
    random_state=42
)

print(training_data.shape)

(50000, 17)


6. Building the TVAE Model

In [7]:
tvae = TVAESynthesizer(
    metadata=metadata,
    epochs=100
)

print("TVAE Model Created")

TVAE Model Created


C:\Users\Admin\.jupyter\New folder\envs\synthetic\Lib\site-packages\sdv\single_table\base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\Admin\.jupyter\New folder\envs\synthetic\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


7. training TVAE

In [8]:
tvae.fit(training_data)

print("TVAE Training Completed Successfully!")

  File "C:\Users\Admin\.jupyter\New folder\envs\synthetic\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\.jupyter\New folder\envs\synthetic\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Admin\.jupyter\New folder\envs\synthetic\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\.jupyter\New folder\envs\synthetic\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Admin\.jupyter\New folder\envs\synthetic\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, arg

BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.

In [ ]:
8. Generating the synthetic dataset